In [1]:
import pickle
import csv
import pandas as pd

In [2]:
with open("./data/text/text_us_2005.pkl", "rb") as f:  # "rb" = read in binary mode
    data = pickle.load(f)

In [86]:
data

,date,cik,file_type,rf,mgmt,gvkey,cusip,year
177795,20050103,16099,10Q,,Item 2 Management s Discussion and Analysis of...,6831.0,549282101,2005
177791,20050103,779544,10K,,Item 7. Management's Discussion and Analysis o...,11872.0,040712101,2005
177794,20050103,831641,10K,,Item 7 \n \n Management's Discussion and Analy...,24783.0,88162G103,2005
177790,20050103,866415,10K,,ITEM 7. Management's Discussion and Analysis o...,61721.0,459412102,2005
177793,20050103,1141240,10Q,,Item\n 2 Management s Discussion and Analysis ...,146117.0,53634X100,2005
...,...,...,...,...,...,...,...,...
195708,20051229,1100983,10K,,Item 7. Management s Discussion and Analysis\n...,133506.0,71086E107,2005
195726,20051229,1122668,10K,ITEM 1A. RISK FACTORS\n\n This Report contains...,ITEM 7. MANAGEMENT'S DISCUSSION AND ANALYSIS O...,141007.0,68382T101,2005
195718,20051229,1310094,10K,,ITEM 7. MANAGEMENT S DISCUSSION AND ANALYSIS O...,162956.0,00430L103,2005
195714,20051229,1311396,10K,Item 1A. \n\nRisk Factors ITEM 1A. Risk Factor...,Item\n 7. \n\nManagement s\n Discussion and An...,165666.0,05381A105,2005


In [65]:
lm_dict = {}
with open("./data/text/Loughran-McDonald_MasterDictionary_1993-2024.csv", "r") as f:
    reader = csv.DictReader(f)
    for row in reader:
        word = row["Word"].lower()
        lm_dict[word] = {
            "positive": int(row["Positive"]),
            "negative": int(row["Negative"]),
            "uncertainty": int(row["Uncertainty"]),
            "litigious": int(row["Litigious"]),
            "strong_modal": int(row["Strong_Modal"]),
            "weak_modal": int(row["Weak_Modal"]),
            "constraining": int(row["Constraining"])
        }


In [77]:
import re
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# Make sure you have NLTK resources downloaded
import nltk
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess_text(text, remove_stopwords=False, lemmatize=True):
    # Lowercase
    text = text.lower()
    
    # Remove numbers & punctuation
    text = re.sub(r'\d+', '', text)
    text = re.sub(r'\W+', ' ', text)
    
    # Tokenize
    tokens = word_tokenize(text)
    
    # Remove stopwords
    if remove_stopwords:
        tokens = [t for t in tokens if t not in stop_words]
    
    # Lemmatize
    if lemmatize:
        tokens = [lemmatizer.lemmatize(t) for t in tokens]
    
    return tokens




[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\shoai\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\shoai\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\shoai\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [80]:
from collections import Counter

def lm_sentiment(tokens):
    counts = Counter()
    for token in tokens:
        if token in lm_dict:
            for category in lm_dict[token]:
                counts[category] += lm_dict[token][category]
    
    total_words = len(tokens) if len(tokens) > 0 else 1  # avoid division by zero
    
    # Compute ratios
    return {
        "positive_ratio": counts["positive"] / total_words,
        "negative_ratio": counts["negative"] / total_words,
        "net_sentiment": (counts["positive"] - counts["negative"]) / total_words,
        "uncertainty_ratio": counts["uncertainty"] / total_words,
        "litigious_ratio": counts["litigious"] / total_words,
        "strong_modal_ratio": counts["strong_modal"] / total_words,
        "weak_modal_ratio": counts["weak_modal"] / total_words,
        "constraining_ratio": counts["constraining"] / total_words
    }


In [87]:
# -----------------------------
# 4. Apply LM pipeline with gvkey
# -----------------------------
def run_lm_pipeline(filings_df, remove_stopwords=True):
    results = []
    gvkeys = []
    
    for i, row in enumerate(filings_df.itertuples(index=False)):
        filing_text = row.mgmt
        gvkey = row.gvkey
        print(f"Processing filing {i+1}/{len(filings_df)}", end='\r')
        
        tokens = preprocess_text(filing_text, remove_stopwords=remove_stopwords)
        sentiment = lm_sentiment(tokens)
        results.append(sentiment)
        gvkeys.append(gvkey)
        
    df = pd.DataFrame(results)
    df.insert(0, 'gvkey', gvkeys)  # add gvkey as the first column
    return df

# Run both versions
sentiment_no_stopwords = run_lm_pipeline(data[["gvkey","mgmt"]], remove_stopwords=False)
sentiment_with_stopwords = run_lm_pipeline(data[["gvkey","mgmt"]], remove_stopwords=True)

# -----------------------------
# 5. Combine for comparison
# -----------------------------
comparison_df = sentiment_no_stopwords.merge(
    sentiment_with_stopwords.add_suffix('_stop'),
    left_on='gvkey',
    right_on='gvkey_stop'
)

# Optional: rename gvkey_stop column to gvkey and drop duplicate if needed
comparison_df.rename(columns={'gvkey_stop':'gvkey'}, inplace=True)
comparison_df.drop(columns=['gvkey'], inplace=False)



print("Done! DataFrame keyed by gvkey ready for analysis.")



Done! DataFrame keyed by gvkey ready for analysis.


In [3]:
import torch

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

Torch version: 2.1.1
CUDA available: False


In [2]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
import torch.nn.functional as F

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# Load tokenizer + model
model_name = "nlpaueb/sec-bert-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)
model.to(device)
model.eval()

# Example usage
text = "The company had a strong quarter with higher revenue."
inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=512)
inputs = {k: v.to(device) for k, v in inputs.items()}  # send tensors to GPU

# Forward pass
with torch.no_grad():
    outputs = model(**inputs)
    probs = F.softmax(outputs.logits, dim=1)  # pos/neu/neg probabilities
    print("Probabilities (pos, neu, neg):", probs)


Using device: cpu


ValueError: Due to a serious vulnerability issue in `torch.load`, even with `weights_only=True`, we now require users to upgrade torch to at least v2.6 in order to use the function. This version restriction does not apply when loading files with safetensors.
See the vulnerability report here https://nvd.nist.gov/vuln/detail/CVE-2025-32434